# Notebook 2 — 3D EGNN + Conformer-Ensemble Pooling on QM9

**Project:** Equivariant Graph Neural Networks with Chirality Encoding for Structure-Sensitive Molecular Property Prediction

**Goal of this notebook:**
1. Generate multiple low-energy 3D conformers per molecule using RDKit's **ETKDGv3** + MMFF94
   optimization.
2. Build an **Equivariant GNN (EGNN)** (Satorras et al., 2021) that uses atomic coordinates and
   interatomic distances as continuous geometric edge information, and whose predictions are
   provably unchanged under rotation/translation of the input molecule.
3. Combine predictions from several conformers per molecule via **conformer-ensemble pooling**
   (average the invariant graph embeddings before the final regression head).
4. Compare the resulting test MAE against the 2D baseline from Notebook 1, on the *same*
   train/val/test molecules.

**Before running this notebook:** upload `qm9_split.pt` and `target_norm_stats.pt` from
Notebook 1 into this Colab's working directory (Files panel → upload). If they're missing, this
notebook regenerates an equivalent split using the same seed, but uploading the real files
guarantees byte-identical splits.


In [ ]:
!pip install -q torch_geometric
!pip install -q rdkit


In [ ]:
import os
import json
import pickle
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader as TorchDataLoader
from torch_geometric.datasets import QM9
from torch_geometric.data import Data, Batch
from torch_geometric.nn import global_mean_pool
from tqdm.auto import tqdm

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')  # silence RDKit's embedding/warning spam

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)


## 1. Load QM9 and reuse Notebook 1's split


In [ ]:
DATA_ROOT = './data/QM9'
dataset = QM9(root=DATA_ROOT)

SPLIT_PATH = 'qm9_split.pt'
NORM_PATH = 'target_norm_stats.pt'

TARGET_NAMES = [
    'mu', 'alpha', 'homo', 'lumo', 'gap', 'r2', 'zpve', 'u0', 'u298',
    'h298', 'g298', 'cv', 'u0_atom', 'u298_atom', 'h298_atom',
    'g298_atom', 'A', 'B', 'C'
]

if os.path.exists(SPLIT_PATH):
    split = torch.load(SPLIT_PATH)
    train_idx, val_idx, test_idx = split['train_idx'], split['val_idx'], split['test_idx']
    TARGET_IDX = split['target_idx']
    print("Loaded split from Notebook 1 (qm9_split.pt) — same molecules, fair comparison.")
else:
    print("qm9_split.pt not found — regenerating an equivalent split with the same seed/logic.")
    TARGET_IDX = 4
    NUM_SAMPLES = 20000
    perm = torch.randperm(len(dataset), generator=torch.Generator().manual_seed(SEED))
    subset_indices = perm[:NUM_SAMPLES]
    n_train = int(0.8 * NUM_SAMPLES)
    n_val = int(0.1 * NUM_SAMPLES)
    train_idx = subset_indices[:n_train]
    val_idx = subset_indices[n_train:n_train + n_val]
    test_idx = subset_indices[n_train + n_val:]

print(f"Predicting target: {TARGET_NAMES[TARGET_IDX]} (index {TARGET_IDX})")
print(f"Full split sizes -> Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

if os.path.exists(NORM_PATH):
    norm_stats = torch.load(NORM_PATH)
    target_mean, target_std = norm_stats['mean'], norm_stats['std']
    print("Loaded normalization stats from Notebook 1 (target_norm_stats.pt).")
else:
    train_targets = torch.stack([dataset[i].y[0, TARGET_IDX] for i in train_idx])
    target_mean, target_std = train_targets.mean(), train_targets.std()

print(f"Target mean: {target_mean:.4f}, std: {target_std:.4f}")


## 2. Subsample for conformer generation

Generating several RDKit conformers per molecule (embed + MMFF-optimize + rank by energy) is
far more expensive than the 2D baseline. For a class project we don't need all 20,000 molecules
from Notebook 1 — we draw a smaller subsample **from the same train/val/test partitions**, so
the comparison with the 2D baseline stays fair (no leakage across splits), while keeping
runtime reasonable on a free Colab GPU.

Increase `MAX_TRAIN` / `MAX_VAL` / `MAX_TEST` later once the pipeline is verified to work.


In [ ]:
MAX_TRAIN, MAX_VAL, MAX_TEST = 3000, 500, 500

g = torch.Generator().manual_seed(SEED)
train_idx_3d = train_idx[torch.randperm(len(train_idx), generator=g)[:MAX_TRAIN]]
val_idx_3d = val_idx[torch.randperm(len(val_idx), generator=g)[:MAX_VAL]]
test_idx_3d = test_idx[torch.randperm(len(test_idx), generator=g)[:MAX_TEST]]

print(f"3D subset -> Train: {len(train_idx_3d)}, Val: {len(val_idx_3d)}, Test: {len(test_idx_3d)}")


## 3. Conformer generation with RDKit ETKDGv3 + MMFF94

**Design note (important for the viva):** rather than trying to force our generated conformers
to line up atom-index-for-atom-index with the *original* QM9 3D graph (which was parsed from a
different source file and can have a different atom ordering / implicit bookkeeping), we
**rebuild the molecular graph directly from RDKit's own `Mol` object** (parsed from SMILES,
explicit hydrogens added). This guarantees the node ordering and bond connectivity are
self-consistent across every conformer of a given molecule, which is all we actually need for
conformer-ensemble pooling — we never need to match against QM9's original `edge_index`.

For each molecule we embed `num_attempts` candidate conformers, optimize each with MMFF94, keep
the `N_CONFORMERS` lowest-energy ones (a simple proxy for "thermodynamically representative"
low-energy geometries).


In [ ]:
N_CONFORMERS = 3  # number of low-energy conformers per molecule


def generate_conformers(smiles, num_confs=N_CONFORMERS, num_attempts=20):
    """Generate `num_confs` low-energy 3D conformers for a molecule.

    Returns (list_of_coordinate_arrays, rdkit_mol) or None on failure. A
    small fraction of QM9 SMILES fail to embed / optimize — these are
    skipped and counted later.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = Chem.AddHs(mol)

    params = AllChem.ETKDGv3()
    params.randomSeed = SEED
    conf_ids = AllChem.EmbedMultipleConfs(mol, numConfs=max(num_confs, num_attempts), params=params)
    if len(conf_ids) == 0:
        return None

    energies = []
    for cid in conf_ids:
        try:
            props = AllChem.MMFFGetMoleculeProperties(mol)
            ff = AllChem.MMFFGetMoleculeForceField(mol, props, confId=cid)
            ff.Minimize()
            energies.append((cid, ff.CalcEnergy()))
        except Exception:
            energies.append((cid, float('inf')))

    energies.sort(key=lambda t: t[1])
    best_conf_ids = [cid for cid, e in energies[:num_confs] if np.isfinite(e)]
    if len(best_conf_ids) == 0:
        return None

    conformers = []
    for cid in best_conf_ids:
        conf = mol.GetConformer(cid)
        conformers.append(conf.GetPositions())  # (num_atoms, 3), explicit Hs included

    # Pad with repeats if fewer than num_confs succeeded (rare but possible)
    while len(conformers) < num_confs:
        conformers.append(conformers[-1])

    return conformers, mol


## 4. Build PyG `Data` objects from the RDKit molecule

Node features: one-hot atomic type (QM9 only contains H, C, N, O, F). Edge features: bond-type
one-hot (single/double/triple/aromatic), built from RDKit's own bonds. `pos` holds the 3D
coordinates for that specific conformer.


In [ ]:
ATOM_TYPES = ['H', 'C', 'N', 'O', 'F']  # the only elements present in QM9
ATOM_TO_IDX = {a: i for i, a in enumerate(ATOM_TYPES)}
BOND_TYPES = [Chem.BondType.SINGLE, Chem.BondType.DOUBLE, Chem.BondType.TRIPLE, Chem.BondType.AROMATIC]


def mol_to_data(mol, coords, y_value):
    atoms = mol.GetAtoms()
    x = torch.zeros(len(atoms), len(ATOM_TYPES))
    for atom in atoms:
        idx = ATOM_TO_IDX.get(atom.GetSymbol())
        if idx is None:
            return None  # unexpected element outside QM9's H/C/N/O/F -> skip molecule
        x[atom.GetIdx(), idx] = 1.0

    edge_index = []
    edge_attr = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bt = bond.GetBondType()
        bt_onehot = [1.0 if bt == b else 0.0 for b in BOND_TYPES]
        edge_index += [[i, j], [j, i]]
        edge_attr += [bt_onehot, bt_onehot]

    if len(edge_index) == 0:
        return None

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    pos = torch.tensor(coords, dtype=torch.float)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, pos=pos,
                y=torch.tensor([y_value], dtype=torch.float))


## 5. Build (and cache) the conformer datasets

This is the slow step. Results are cached to disk (`conformers_*.pkl`) so re-running the
notebook (e.g. after a Colab disconnect) doesn't redo the RDKit work.


In [ ]:
def build_conformer_dataset(indices, target_idx, mean, std, cache_path):
    if os.path.exists(cache_path):
        print(f"Loading cached conformer dataset from {cache_path}")
        with open(cache_path, 'rb') as f:
            return pickle.load(f)

    samples = []  # each entry: list of N_CONFORMERS Data objects for one molecule
    skipped = 0
    for idx in tqdm(indices.tolist(), desc=f"Generating conformers -> {cache_path}"):
        data = dataset[idx]
        smiles = getattr(data, 'smiles', None)
        if smiles is None:
            skipped += 1
            continue

        result = generate_conformers(smiles)
        if result is None:
            skipped += 1
            continue
        conf_coords_list, mol = result

        y_raw = data.y[0, target_idx].item()
        y_norm = (y_raw - mean.item()) / std.item()

        mol_data_list = []
        ok = True
        for coords in conf_coords_list:
            d = mol_to_data(mol, coords, y_norm)
            if d is None:
                ok = False
                break
            mol_data_list.append(d)
        if not ok:
            skipped += 1
            continue

        samples.append(mol_data_list)

    print(f"Built {len(samples)} molecules, skipped {skipped} "
          f"(failed SMILES parse / conformer embedding / unexpected atom type).")
    with open(cache_path, 'wb') as f:
        pickle.dump(samples, f)
    return samples


train_conf_data = build_conformer_dataset(train_idx_3d, TARGET_IDX, target_mean, target_std, 'conformers_train.pkl')
val_conf_data = build_conformer_dataset(val_idx_3d, TARGET_IDX, target_mean, target_std, 'conformers_val.pkl')
test_conf_data = build_conformer_dataset(test_idx_3d, TARGET_IDX, target_mean, target_std, 'conformers_test.pkl')


## 6. Dataset + collate function for conformer batches

Each molecule contributes `N_CONFORMERS` graphs (same topology, different `pos`). The custom
collate function groups conformer *k* of every molecule in the minibatch into its own PyG
`Batch`, so we get a list of `N_CONFORMERS` batches to feed through the shared encoder.


In [ ]:
class ConformerEnsembleDataset(torch.utils.data.Dataset):
    def __init__(self, samples):
        self.samples = samples  # list of lists of Data (len == N_CONFORMERS each)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        return self.samples[i]


def collate_conformers(batch):
    """batch: list of molecules, each a list of N_CONFORMERS Data objects.
    Returns a list of N_CONFORMERS PyG Batch objects: conformer_batches[k]
    holds the k-th conformer of every molecule in this minibatch, in the
    same molecule order for every k. Running the same encoder on each of
    these and averaging the resulting graph embeddings is the
    conformer-ensemble pooling step.
    """
    n_conf = len(batch[0])
    conformer_batches = []
    for k in range(n_conf):
        graphs_k = [mol[k] for mol in batch]
        conformer_batches.append(Batch.from_data_list(graphs_k))
    return conformer_batches


train_loader_3d = TorchDataLoader(ConformerEnsembleDataset(train_conf_data), batch_size=32,
                                   shuffle=True, collate_fn=collate_conformers)
val_loader_3d = TorchDataLoader(ConformerEnsembleDataset(val_conf_data), batch_size=64,
                                 shuffle=False, collate_fn=collate_conformers)
test_loader_3d = TorchDataLoader(ConformerEnsembleDataset(test_conf_data), batch_size=64,
                                  shuffle=False, collate_fn=collate_conformers)


## 7. EGNN layer (Satorras et al., 2021)

- **Messages** depend on node features and the *squared distance* between connected atoms —
  invariant to rotating/translating the whole molecule.
- **Coordinate update** moves each atom along the (equivariant) direction vectors to its
  neighbours, scaled by a learned scalar. Rotate/translate the input and the updated
  coordinates rotate/translate identically — this is what makes the layer *equivariant*.
- **Node feature update** aggregates messages (invariant) into a new invariant hidden state.


In [ ]:
class EGNNLayer(nn.Module):
    """One E(n)-Equivariant Graph Neural Network layer."""

    def __init__(self, hidden_dim, edge_feat_dim):
        super().__init__()
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 1 + edge_feat_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
        )
        self.coord_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1),
        )
        self.node_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, h, pos, edge_index, edge_attr):
        row, col = edge_index
        rel_pos = pos[row] - pos[col]
        dist_sq = (rel_pos ** 2).sum(dim=-1, keepdim=True)  # invariant scalar

        edge_input = torch.cat([h[row], h[col], dist_sq, edge_attr], dim=-1)
        m_ij = self.edge_mlp(edge_input)

        # Equivariant coordinate update: weighted sum of relative direction vectors
        coord_weight = self.coord_mlp(m_ij)
        coord_update = rel_pos * coord_weight
        agg_coord = torch.zeros_like(pos)
        agg_coord.index_add_(0, row, coord_update)
        deg = torch.zeros(pos.size(0), 1, device=pos.device)
        deg.index_add_(0, row, torch.ones_like(coord_weight))
        pos_new = pos + agg_coord / deg.clamp(min=1.0)

        # Invariant node-feature update
        agg_m = torch.zeros_like(h)
        agg_m.index_add_(0, row, m_ij)
        h_new = h + self.node_mlp(torch.cat([h, agg_m], dim=-1))

        return h_new, pos_new


## 8. Encoder + conformer-ensemble model

The encoder ends in `global_mean_pool` over the final (invariant) node features, so its output
is an invariant per-molecule embedding for one conformer. The ensemble model runs the *same*
encoder once per conformer and **averages the embeddings** before the regression head — this is
the conformer-ensemble pooling described in the project plan.


In [ ]:
class EGNNEncoder(nn.Module):
    def __init__(self, node_in_dim, edge_in_dim, hidden_dim=64, num_layers=4):
        super().__init__()
        self.embed = nn.Linear(node_in_dim, hidden_dim)
        self.layers = nn.ModuleList([EGNNLayer(hidden_dim, edge_in_dim) for _ in range(num_layers)])

    def forward(self, x, pos, edge_index, edge_attr, batch):
        h = self.embed(x)
        for layer in self.layers:
            h, pos = layer(h, pos, edge_index, edge_attr)
        return global_mean_pool(h, batch)  # invariant graph embedding


class EGNNConformerEnsemble(nn.Module):
    """Shared EGNN encoder run once per conformer; embeddings are averaged
    across conformers (conformer-ensemble pooling) before the regression
    head."""

    def __init__(self, node_in_dim, edge_in_dim, hidden_dim=64, num_layers=4):
        super().__init__()
        self.encoder = EGNNEncoder(node_in_dim, edge_in_dim, hidden_dim, num_layers)
        self.readout = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, conformer_batches):
        embeddings = []
        for batch_k in conformer_batches:
            emb_k = self.encoder(batch_k.x, batch_k.pos, batch_k.edge_index,
                                  batch_k.edge_attr, batch_k.batch)
            embeddings.append(emb_k)
        ensemble_embedding = torch.stack(embeddings, dim=0).mean(dim=0)
        out = self.readout(ensemble_embedding)
        return out.view(-1)


In [ ]:
node_in_dim = len(ATOM_TYPES)
edge_in_dim = len(BOND_TYPES)

model_3d = EGNNConformerEnsemble(node_in_dim, edge_in_dim).to(device)
print(model_3d)


## 9. Training utilities


In [ ]:
def train_one_epoch_3d(model, loader, optimizer, device):
    model.train()
    total_loss, n = 0.0, 0
    for conformer_batches in loader:
        conformer_batches = [b.to(device) for b in conformer_batches]
        y = conformer_batches[0].y  # identical target repeated across conformer slots
        optimizer.zero_grad()
        pred = model(conformer_batches)
        loss = F.l1_loss(pred, y)
        loss.backward()
        optimizer.step()
        bs = y.size(0)
        total_loss += loss.item() * bs
        n += bs
    return total_loss / n


@torch.no_grad()
def evaluate_3d(model, loader, device, mean, std):
    model.eval()
    total_mae_norm, total_mae_real, n = 0.0, 0.0, 0
    for conformer_batches in loader:
        conformer_batches = [b.to(device) for b in conformer_batches]
        y = conformer_batches[0].y
        pred = model(conformer_batches)
        total_mae_norm += F.l1_loss(pred, y, reduction='sum').item()
        pred_real = pred * std.to(device) + mean.to(device)
        y_real = y * std.to(device) + mean.to(device)
        total_mae_real += F.l1_loss(pred_real, y_real, reduction='sum').item()
        n += y.size(0)
    return total_mae_norm / n, total_mae_real / n


## 10. Train


In [ ]:
optimizer_3d = torch.optim.Adam(model_3d.parameters(), lr=1e-3)
scheduler_3d = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_3d, mode='min', factor=0.5, patience=5)

N_EPOCHS_3D = 60
best_val_mae_3d = float('inf')
history_3d = {'train_loss': [], 'val_mae_norm': [], 'val_mae_real': []}

for epoch in range(1, N_EPOCHS_3D + 1):
    train_loss = train_one_epoch_3d(model_3d, train_loader_3d, optimizer_3d, device)
    val_mae_norm, val_mae_real = evaluate_3d(model_3d, val_loader_3d, device, target_mean, target_std)
    scheduler_3d.step(val_mae_norm)

    history_3d['train_loss'].append(train_loss)
    history_3d['val_mae_norm'].append(val_mae_norm)
    history_3d['val_mae_real'].append(val_mae_real)

    if val_mae_norm < best_val_mae_3d:
        best_val_mae_3d = val_mae_norm
        torch.save(model_3d.state_dict(), 'egnn_3d_best.pt')

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} | Train L1 (norm): {train_loss:.4f} | "
              f"Val MAE (norm): {val_mae_norm:.4f} | Val MAE (real units): {val_mae_real:.4f}")


## 11. Test evaluation


In [ ]:
model_3d.load_state_dict(torch.load('egnn_3d_best.pt'))
test_mae_norm_3d, test_mae_real_3d = evaluate_3d(model_3d, test_loader_3d, device, target_mean, target_std)
print(f"Test MAE (normalized): {test_mae_norm_3d:.4f}")
print(f"Test MAE (real units, {TARGET_NAMES[TARGET_IDX]}): {test_mae_real_3d:.4f}")


## 12. Sanity check — verify rotation/translation invariance

This directly demonstrates the property the EGNN architecture is designed to guarantee: apply a
random rotation + translation to every conformer's coordinates and confirm the prediction is
essentially unchanged (up to floating-point error). Good evidence to show in the viva.


In [ ]:
@torch.no_grad()
def check_invariance(model, loader, device, n_batches=1):
    model.eval()
    for b_idx, conformer_batches in enumerate(loader):
        conformer_batches = [b.to(device) for b in conformer_batches]

        # Random rotation matrix (orthogonal, via QR decomposition) + random translation
        Q, _ = torch.linalg.qr(torch.randn(3, 3, device=device))
        translation = torch.randn(1, 3, device=device) * 5.0

        pred_original = model(conformer_batches)

        rotated_batches = []
        for batch_k in conformer_batches:
            batch_k_rot = batch_k.clone()
            batch_k_rot.pos = batch_k.pos @ Q + translation
            rotated_batches.append(batch_k_rot)
        pred_rotated = model(rotated_batches)

        max_diff = (pred_original - pred_rotated).abs().max().item()
        print(f"Batch {b_idx}: max |pred_original - pred_rotated| = {max_diff:.6f} "
              f"(should be ~0)")

        if b_idx + 1 >= n_batches:
            break


check_invariance(model_3d, test_loader_3d, device)


## 13. Save results and compare with the 2D baseline


In [ ]:
results_3d = {
    'model_name': '3D_EGNN_conformer_ensemble',
    'target': TARGET_NAMES[TARGET_IDX],
    'target_idx': TARGET_IDX,
    'num_conformers': N_CONFORMERS,
    'num_train_molecules': len(train_conf_data),
    'num_val_molecules': len(val_conf_data),
    'num_test_molecules': len(test_conf_data),
    'test_mae_normalized': test_mae_norm_3d,
    'test_mae_real_units': test_mae_real_3d,
}

with open('results_3d_egnn.json', 'w') as f:
    json.dump(results_3d, f, indent=2)

print(json.dumps(results_3d, indent=2))

if os.path.exists('results_2d_baseline.json'):
    with open('results_2d_baseline.json') as f:
        results_2d = json.load(f)
    print()
    print(f"2D GNN baseline   test MAE ({results_2d['target']}): {results_2d['test_mae_real_units']:.4f}")
    print(f"3D EGNN ensemble  test MAE ({results_3d['target']}): {results_3d['test_mae_real_units']:.4f}")
    print("(Note: the 3D model is trained/evaluated on a smaller molecule subset — "
          "treat this as a first-pass comparison, not a final number.)")
else:
    print("Upload results_2d_baseline.json (from Notebook 1) to see a side-by-side comparison here.")


## 14. Training curve


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.plot(history_3d['train_loss'], label='Train L1 loss (normalized)')
plt.plot(history_3d['val_mae_norm'], label='Val MAE (normalized)')
plt.xlabel('Epoch')
plt.ylabel('Loss / MAE')
plt.title('3D EGNN + Conformer Ensemble — Training Curve')
plt.legend()
plt.show()


## Done — files to keep for later notebooks

- `results_3d_egnn.json` — this model's test MAE, for the final comparison table
- `egnn_3d_best.pt` — trained weights (optional, if you want to reload without retraining)
- `conformers_train.pkl` / `conformers_val.pkl` / `conformers_test.pkl` — cached conformers
  (can be large; regenerate instead of re-uploading if storage is tight)

**Next up (Notebook 3):** add explicit chirality (R/S) and double-bond stereochemistry (E/Z) to
the node/edge features and message passing, and test whether the model can now tell enantiomers
and diastereomers apart.
